# Homework 12

In [1]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from statsmodels.formula.api import ols

In [2]:
df1 = pd.read_csv('homework_12.1.csv', index_col=0)
df2 = pd.read_csv('homework_12.2.csv', index_col=0)

In [4]:
print(df1.head(10))

          Y      Time  Group
0 -0.232900 -1.193204      0
1  2.848846 -1.607748      1
2  0.550209 -0.269793      0
3  2.198280  7.743730      0
4  4.111044 -4.244359      1
5  1.153886 -3.843564      0
6 -0.515597 -5.921128      0
7  1.714020 -2.013572      0
8  1.565405 -5.493066      0
9  1.149523  1.198473      0


## Question 1

In [5]:
import pandas as pd
import statsmodels.formula.api as smf

# Assuming df1 is already defined
# Create treatment-time interaction
df1['Post'] = (df1['Time'] > 0).astype(int)
df1['Treatment'] = (df1['Group'] == 1).astype(int)
df1['DiD'] = df1['Post'] * df1['Treatment']

# Difference-in-Differences regression
model = smf.ols('Y ~ Treatment + Post + DiD', data=df1).fit()
print(model.summary())

# Extract the treatment effect (DiD coefficient)
effect = model.params['DiD']
print("Estimated treatment effect (DiD):", effect)

                            OLS Regression Results                            
Dep. Variable:                      Y   R-squared:                       0.638
Model:                            OLS   Adj. R-squared:                  0.638
Method:                 Least Squares   F-statistic:                     5866.
Date:                Sun, 07 Dec 2025   Prob (F-statistic):               0.00
Time:                        14:46:20   Log-Likelihood:                -14214.
No. Observations:               10000   AIC:                         2.844e+04
Df Residuals:                    9996   BIC:                         2.846e+04
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept      0.9802      0.016     61.953      0.0

In [6]:
# Define pre/post periods
df1['Post'] = (df1['Time'] > 0).astype(int)

# Group means
group_means = df1.groupby(['Group', 'Post'])['Y'].mean().unstack()

# Extract means
A = group_means.loc[0, 0]  # Control, Pre
B = group_means.loc[0, 1]  # Control, Post
C = group_means.loc[1, 0]  # Treatment, Pre
D = group_means.loc[1, 1]  # Treatment, Post

# Manual DiD
manual_effect = (D - C) - (B - A)
print("Manual DiD estimate:", manual_effect)

Manual DiD estimate: 0.9440103783528997


## Question 2

In [7]:
import statsmodels.formula.api as smf

# Filter to pre-treatment data
pre_df = df2[df2['Time'] <= 0].copy()

# Create interaction term
pre_df['Group'] = pre_df['Group'].astype(int)
pre_df['GroupTime'] = pre_df['Group'] * pre_df['Time']

# Run regression: Y ~ Group + Time + Group*Time
model_pre = smf.ols('Y ~ Group + Time + GroupTime', data=pre_df).fit()
print(model_pre.summary())

# Extract t-value of interaction term
t_value = model_pre.tvalues['GroupTime']
print("t-value of Group × Time interaction:", t_value)

                            OLS Regression Results                            
Dep. Variable:                      Y   R-squared:                       0.324
Model:                            OLS   Adj. R-squared:                  0.324
Method:                 Least Squares   F-statistic:                     1591.
Date:                Sun, 07 Dec 2025   Prob (F-statistic):               0.00
Time:                        14:48:40   Log-Likelihood:                -13981.
No. Observations:                9944   AIC:                         2.797e+04
Df Residuals:                    9940   BIC:                         2.800e+04
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept      0.9996      0.022     45.154      0.0

## Question 3

In [8]:
import numpy as np
import statsmodels.api as sm

def generate_ar1(n, rho):
    eps = np.random.normal(0, np.sqrt(1 - rho**2), size=n)
    x = np.zeros(n)
    for t in range(1, n):
        x[t] = rho * x[t-1] + eps[t]
    return x

n = 10000
rho = 0.8
n_trials = 1000
estimates = []

for _ in range(n_trials):
    X = generate_ar1(n, rho)
    error = generate_ar1(n, rho)
    Y = 2 * X + error
    model = sm.OLS(Y, sm.add_constant(X)).fit()
    estimates.append(model.params[1])  # Coefficient on X

simulated_se = np.std(estimates)
print("Simulated standard error:", simulated_se)

Simulated standard error: 0.021295064746356878


## Question 4

In [15]:
import numpy as np
import statsmodels.api as sm
from sklearn.linear_model import LinearRegression

np.random.seed(50)
X1 = np.random.normal(0, 1, 1000)
X2 = np.random.normal(0, 1, 1000) + X1
X3 = np.random.normal(0, 1, 1000) + 2 * X2

# Regress X1 on X2 and X3
X = np.column_stack((X2, X3))
model = LinearRegression().fit(X, X1)
r_squared = model.score(X, X1)

# Compute VIF
vif = 1 / (1 - r_squared)
print("VIF for X1:", vif)

VIF for X1: 2.03936439455897


In [16]:
import numpy as np
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor

np.random.seed(50)
X1 = np.random.normal(0, 1, 1000)
X2 = np.random.normal(0, 1, 1000) + X1
X3 = np.random.normal(0, 1, 1000) + 2 * X2

X = np.column_stack([X1, X2, X3])
X_const = sm.add_constant(X)

vifs = [variance_inflation_factor(X_const, i) for i in range(X_const.shape[1])]
print({"const": vifs[0], "VIF_X1": vifs[1], "VIF_X2": vifs[2], "VIF_X3": vifs[3]})

{'const': np.float64(1.0030247851021459), 'VIF_X1': np.float64(2.0393643945589695), 'VIF_X2': np.float64(10.753125479599182), 'VIF_X3': np.float64(9.853977615806759)}


In [11]:
from sklearn.linear_model import LinearRegression

X_reg = np.column_stack([X2, X3])
r2 = LinearRegression().fit(X_reg, X1).score(X_reg, X1)
vif_manual = 1 / (1 - r2)
print("Manual VIF for X1:", vif_manual)

Manual VIF for X1: 1.9783742480901259


In [12]:
import numpy as np

corr = np.corrcoef(np.column_stack([X1, X2, X3]).T)
eigvals = np.linalg.eigvalsh(corr)
cond_num = np.sqrt(eigvals.max() / eigvals.min())
print("Correlation matrix:\n", corr)
print("Correlation condition number:", cond_num)

Correlation matrix:
 [[1.         0.70299715 0.6566582 ]
 [0.70299715 1.         0.94269849]
 [0.6566582  0.94269849 1.        ]]
Correlation condition number: 6.802152965053287
